# LP-ResMIL: fully nested participant-level evaluation

**Only the new LP-ResMIL model is trained here.** The previously evaluated LPGNet baseline,
Variant A, and Variant B are not rerun. Their completed recording-level OOF predictions can
optionally be loaded later for paired comparisons.

LP-ResMIL combines fold-local order-11 linear-prediction residuals, a compact multi-scale
residual encoder, and gated attention multiple-instance learning (MIL) over windows. Unlike
window-only training, the recording stage is optimized using one diagnosis per recording;
unlike the frozen LPGNet second stage, nested selection can either freeze the encoder or
fine-tune its final residual block.

## Evaluation contract

For every declared training seed and outer fold:

1. Outer-test participants are excluded from LP fitting, normalization, candidate search,
   early stopping, epoch selection, and final refitting.
2. All candidates share the same inner participant splits and deterministic candidate list.
3. Inner preprocessing is refit using inner-training controls only.
4. Candidates are ranked by pooled inner **participant log loss**, with AUC only as a
   tie-breaker. Window, recording, and participant partitions remain distinct.
5. Window and recording losses are weighted so each participant contributes equally within
   the corresponding fitting partition.
6. The selected configuration is refit on all outer-training participants for epochs chosen
   using held-out outer-training participants, then evaluated once on outer-test recordings.
7. Participant OOF predictions are primary. Each participant's recording probabilities are
   averaged; recording-level metrics are secondary. The decision threshold is fixed at 0.5.

The default uses three training seeds and 12 unique candidates per outer fold and seed. This
is computationally expensive. Any reduced run is developmental and must not be reported as
the complete publication experiment.

In [ ]:
# ============================================================
# Reproducibility, imports, and immutable protocol constants
# ============================================================
import os, time, random, json, urllib.request, zipfile, warnings, platform
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy, sklearn, tensorflow as tf
from scipy.signal import lfilter
from IPython.display import display
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             log_loss, roc_curve)
from sklearn.calibration import calibration_curve
from tensorflow import keras
from tensorflow.keras import layers

SEED = 1
FOLD_SEED = 0                 # identical to the preceding LPGNet notebooks
SPLIT_SEED = 9137             # inner/early-stop splits; independent of training seed
SEARCH_SEED = 20261117        # candidate lists; independent of training seed
N_FOLDS, N_INNER = 10, 3
VAL_FRAC = 0.15
LP_ORDER = 11
WIN, STRIDE = 100, 50         # 2 s / 1 s at 50 Hz
BATCH_WIN, BATCH_REC, PRED_BATCH = 128, 8, 8
LR_WIN = 5e-4
LABEL_SMOOTH = 0.1
MAX_EPOCHS_WIN, MAX_EPOCHS_MIL = 1000, 200
PATIENCE, RLR_PATIENCE, MIN_LR = 25, 14, 1e-7
THRESHOLD = 0.5
OUTDIR = Path('lp_resmil_nested_outputs')

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ------------------------ complete publication run ------------------------
TRAIN_SEEDS = [42, 123, 2026]
N_TRIALS = 12                  # unique candidates per outer fold and training seed
N_BOOT = 2000
EXPORT_ATTENTION = True
# Optional completed notebook-2 output; no reference model is retrained.
REFERENCE_OOF_PATH = Path('lpgnet_nested_outputs/recording_oof.csv')

In [ ]:
# ============================================================
# Download PhysioNet data and attach verified clinical metadata
# ============================================================
URL = 'https://physionet.org/static/published-projects/gaitpdb/gait-in-parkinsons-disease-1.0.0.zip'
ZIP = 'gaitpdb.zip'; DATA_DIR = 'gaitpdb_data'
ROOT = os.path.join(DATA_DIR, 'gait-in-parkinsons-disease-1.0.0')
if not os.path.isdir(ROOT):
    if not os.path.exists(ZIP):
        print('Downloading PhysioNet gaitpdb (~300 MB, one-off)...')
        urllib.request.urlretrieve(URL, ZIP)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(DATA_DIR)

records = []
for fn in sorted(os.listdir(ROOT)):
    if not fn.endswith('.txt') or fn[2:4] not in ('Co', 'Pt'):
        continue
    base = fn[:-4]
    arr = np.loadtxt(os.path.join(ROOT, fn), delimiter='\t')
    records.append(dict(file=fn, study=base[:2], subject=base.split('_')[0],
                        label=int(base[2:4] == 'Pt'),
                        signal=arr[:, 1:19].astype(np.float32)))
subjects = sorted({r['subject'] for r in records})
subj_label = {s: next(r['label'] for r in records if r['subject'] == s) for s in subjects}
by_file = {r['file']: i for i, r in enumerate(records)}
by_subject = {s: [i for i, r in enumerate(records) if r['subject'] == s] for s in subjects}
print(f"{len(records)} recordings | {len(subjects)} participants with sensor files | "
      f"{sum(subj_label.values())} PD / {len(subjects)-sum(subj_label.values())} controls")

# demographics.txt is distributed with gaitpdb. Missing values remain missing.
demo_path = os.path.join(ROOT, 'demographics.txt')
demo = pd.read_csv(demo_path, sep='\t', engine='python', usecols=range(20))
demo = demo.loc[:, ~demo.columns.str.startswith('Unnamed')].copy()
demo = demo.dropna(subset=['ID']).copy()
demo['ID'] = demo['ID'].astype(str)
demo = demo.drop_duplicates('ID').set_index('ID')
clinical = {}
for s in subjects:
    row = demo.loc[s] if s in demo.index else None
    hy = pd.to_numeric(row.get('HoehnYahr'), errors='coerce') if row is not None else np.nan
    # Controls are not assigned HY=0 by inference. Only recorded PD values are used.
    verified_hy = float(hy) if subj_label[s] == 1 and pd.notna(hy) else np.nan
    clinical[s] = dict(
        study=s[:2], diagnosis='PD' if subj_label[s] else 'Control',
        hy=verified_hy, hy_label=(f'{verified_hy:g}' if pd.notna(verified_hy)
                                  else ('Unknown' if subj_label[s] else 'Not applicable')),
        hy_status=('verified' if pd.notna(verified_hy) else
                   ('missing' if subj_label[s] else 'not_applicable')),
        age=pd.to_numeric(row.get('Age'), errors='coerce') if row is not None else np.nan,
        gender=pd.to_numeric(row.get('Gender'), errors='coerce') if row is not None else np.nan)
missing_metadata = [s for s in subjects if s not in demo.index]
print('Participants without a demographics join:', missing_metadata)


In [ ]:
# ============================================================
# Official preprocessing and LP residuals
# ============================================================
def preprocess(x):
    # Causal two-tap moving average, then decimation 100 -> 50 Hz.
    return lfilter([0.5, 0.5], 1, x, axis=0)[::2]

def fit_lp(control_sigs, order=LP_ORDER):
    """One predictor per channel; MATLAB-compatible pooled-control fitting."""
    C, conc_all = np.zeros((18, order)), []
    for c in range(18):
        # generateLPresidual.m starts with zero padding and inserts it between walks.
        pieces = [np.zeros(order, np.float32)]
        pieces += [np.concatenate([s[:, c], np.zeros(order, np.float32)])
                   for s in control_sigs]
        conc = np.concatenate(pieces)
        conc_all.append(conc)
        n = len(conc)
        nfft = 1 << (2*n - 1).bit_length()
        f = np.fft.rfft(conc, nfft)
        r = np.fft.irfft(f*np.conj(f), nfft)[:order+1] / n
        R = np.array([[r[abs(i-j)] for j in range(order)] for i in range(order)])
        C[c] = np.linalg.solve(R, r[1:order+1])
    # MATLAB std uses N-1 normalization. No test recordings enter this calculation.
    stds = np.std(np.stack(conc_all, axis=1), axis=0, ddof=1) + 1e-8
    return C, stds

def apply_lp(x, C, stds):
    e = np.stack([lfilter(np.r_[1.0, -C[c]], 1, x[:, c]) for c in range(18)], axis=1)
    return (e / stds).astype(np.float32)

def lp_features(indices, C, stds):
    return {i: apply_lp(records[i]['proc'], C, stds) for i in indices}

for r in records:
    r['proc'] = preprocess(r['signal'])


In [ ]:
# ============================================================
# Fixed participant-disjoint outer folds and assertions
# ============================================================
def make_folds():
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=FOLD_SEED)
    arr = np.array(subjects); y = np.array([subj_label[s] for s in subjects])
    folds = []
    for tr_s, te_s in skf.split(arr, y):
        tr_subj, te_subj = set(arr[tr_s]), set(arr[te_s])
        assert not tr_subj & te_subj
        tr = [i for i, r in enumerate(records) if r['subject'] in tr_subj]
        te = [i for i, r in enumerate(records) if r['subject'] in te_subj]
        assert not ({records[i]['subject'] for i in tr} & {records[i]['subject'] for i in te})
        folds.append(dict(train=tr, test=te))
    return folds

def split_val(indices, frac=VAL_FRAC, seed=SEED):
    """Split participants, never recordings; output participant sets are disjoint."""
    ss = sorted({records[i]['subject'] for i in indices})
    yy = [subj_label[s] for s in ss]
    if len(set(yy)) > 1:
        splitter = StratifiedShuffleSplit(1, test_size=frac, random_state=seed)
        tr_s, va_s = next(splitter.split(np.zeros(len(ss)), yy))
    else:
        perm = np.random.default_rng(seed).permutation(len(ss))
        n_va = max(1, int(round(frac*len(ss))))
        va_s, tr_s = perm[:n_va], perm[n_va:]
    tr_subj, va_subj = {ss[i] for i in tr_s}, {ss[i] for i in va_s}
    assert not tr_subj & va_subj
    return ([i for i in indices if records[i]['subject'] in tr_subj],
            [i for i in indices if records[i]['subject'] in va_subj])

def labels(indices):
    return np.array([records[i]['label'] for i in indices], dtype=np.float32)

FOLDS = make_folds()
for k, f in enumerate(FOLDS):
    tr_n = len({records[i]['subject'] for i in f['train']})
    te_n = len({records[i]['subject'] for i in f['test']})
    print(f"Fold {k}: {tr_n} train participants / {te_n} test participants | "
          f"{len(f['train'])} train / {len(f['test'])} test recordings")


In [ ]:
# ============================================================
# OOF metrics and participant-level diagnostics
# ============================================================
def metric_dict(y,p):
    y=np.asarray(y,int); p=np.asarray(p,float); yh=(p>=THRESHOLD).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,yh,labels=[0,1]).ravel()
    return dict(n=len(y),accuracy=accuracy_score(y,yh),
                balanced_accuracy=balanced_accuracy_score(y,yh),
                macro_f1=f1_score(y,yh,average='macro',zero_division=0),
                f1_pd=f1_score(y,yh,zero_division=0),
                sensitivity=tp/(tp+fn) if tp+fn else np.nan,
                specificity=tn/(tn+fp) if tn+fp else np.nan,
                roc_auc=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
                log_loss=log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1]),
                brier=float(np.mean((p-y)**2)))

def oof_frame(rows):
    df=pd.DataFrame(rows)
    df['prediction']=(df['probability']>=THRESHOLD).astype(int)
    df['correct']=(df['prediction']==df['label']).astype(int)
    df['confidence']=np.maximum(df['probability'],1-df['probability'])
    df['nll']=-(df['label']*np.log(np.clip(df['probability'],1e-7,1))+
                (1-df['label'])*np.log(np.clip(1-df['probability'],1e-7,1)))
    for col in ('study','diagnosis','hy','hy_label','hy_status','age','gender'):
        df[col]=df['subject'].map(lambda s:clinical[s][col])
    return df

def participant_oof(df,group_cols):
    """Average recording probabilities so each participant has equal weight."""
    keys=group_cols+['subject']
    agg=dict(label=('label','first'),fold=('fold','first'),
             probability=('probability','mean'),recordings=('file','nunique'))
    out=df.groupby(keys,as_index=False).agg(**agg)
    return oof_frame(out.to_dict('records'))

def summarize_oof(df,group_cols):
    rows=[]
    for keys,g in df.groupby(group_cols):
        if not isinstance(keys,tuple): keys=(keys,)
        part=participant_oof(g,group_cols)
        row={**dict(zip(group_cols,keys)),
             **{'recording_'+k:v for k,v in metric_dict(g['label'],g['probability']).items()}}
        row.update({'participant_'+k:v for k,v in metric_dict(part['label'],part['probability']).items()})
        rows.append(row)
    return pd.DataFrame(rows)

def wilson_interval(correct,n,z=1.96):
    if n==0: return (np.nan,np.nan)
    ph=correct/n; den=1+z*z/n
    center=(ph+z*z/(2*n))/den
    half=z*np.sqrt(ph*(1-ph)/n+z*z/(4*n*n))/den
    return center-half,center+half

def error_tables(df,group_cols):
    part=participant_oof(df,group_cols)
    errors=part[part['correct']==0].sort_values(group_cols+['nll'],
             ascending=[True]*len(group_cols)+[False])
    pd_rows=[]
    for keys,g in part[part['label']==1].groupby(group_cols+['hy_label'],dropna=False):
        if not isinstance(keys,tuple): keys=(keys,)
        n=len(g); correct=int(g['correct'].sum()); lo,hi=wilson_interval(correct,n)
        pd_rows.append({**dict(zip(group_cols+['hy_label'],keys)),
                        'pd_participants':n,'correct':correct,'errors':n-correct,
                        'pd_sensitivity':correct/n,'wilson_95_low':lo,'wilson_95_high':hi})
    study_rows=[]
    for keys,g in part.groupby(group_cols+['study','diagnosis']):
        if not isinstance(keys,tuple): keys=(keys,)
        study_rows.append({**dict(zip(group_cols+['study','diagnosis'],keys)),
                           **metric_dict(g['label'],g['probability'])})
    return part,errors,pd.DataFrame(pd_rows),pd.DataFrame(study_rows)

def show_table(df,title,limit=100):
    print('\n'+title); display(df.head(limit))

def pooled_bootstrap(df,group_cols,metrics=('accuracy','balanced_accuracy','macro_f1',
                     'sensitivity','specificity','roc_auc','log_loss','brier')):
    """Diagnosis-stratified participant bootstrap conditional on fitted OOF predictions."""
    rows=[]
    for keys,g in df.groupby(group_cols):
        if not isinstance(keys,tuple): keys=(keys,)
        part=participant_oof(g,group_cols).reset_index(drop=True)
        y=part['label'].to_numpy(); p=part['probability'].to_numpy()
        classes=[np.flatnonzero(y==c) for c in (0,1)]
        for metric in metrics:
            rng=np.random.default_rng(2026+sum(ord(ch) for ch in metric+str(keys))%10000)
            vals=[]
            for _ in range(N_BOOT):
                ii=np.concatenate([rng.choice(ix,len(ix),replace=True) for ix in classes])
                vals.append(metric_dict(y[ii],p[ii])[metric])
            rows.append({**dict(zip(group_cols,keys)),'metric':metric,
                         'estimate':metric_dict(y,p)[metric],
                         'bootstrap_95_low':np.nanpercentile(vals,2.5),
                         'bootstrap_95_high':np.nanpercentile(vals,97.5)})
    return pd.DataFrame(rows)

def plot_oof_diagnostics(df,group_cols):
    part=participant_oof(df,group_cols); groups=list(part.groupby(group_cols))
    fig,axes=plt.subplots(len(groups),3,figsize=(16,4.3*len(groups)),squeeze=False)
    for row,(keys,g) in enumerate(groups):
        if not isinstance(keys,tuple): keys=(keys,)
        title=' | '.join(f'{c}={v}' for c,v in zip(group_cols,keys))
        cm=confusion_matrix(g['label'],g['prediction'],labels=[0,1])
        ConfusionMatrixDisplay(cm,display_labels=['Control','PD']).plot(
            ax=axes[row,0],colorbar=False)
        axes[row,0].set_title(title+' | participant OOF')
        fpr,tpr,_=roc_curve(g['label'],g['probability'])
        axes[row,1].plot(fpr,tpr,label=f"AUC={roc_auc_score(g['label'],g['probability']):.3f}")
        axes[row,1].plot([0,1],[0,1],'--',color='gray')
        axes[row,1].set(xlabel='False-positive rate',ylabel='True-positive rate',title=title)
        axes[row,1].legend()
        frac,meanp=calibration_curve(g['label'],g['probability'],n_bins=8,strategy='quantile')
        axes[row,2].plot(meanp,frac,'o-',label='Observed')
        axes[row,2].plot([0,1],[0,1],'--',color='gray')
        axes[row,2].set(xlabel='Mean predicted probability',ylabel='Observed PD fraction',
                        xlim=(0,1),ylim=(0,1),title='Calibration (descriptive)')
        axes[row,2].legend()
    fig.tight_layout(); plt.show()

## Model and search space

The window encoder has three parallel separable-convolution branches, a channel projection,
three dilated residual blocks with squeeze–excitation, and temporal attention within each
window. The recording model applies the same encoder to every valid window, then uses masked
gated attention across windows. Zero-padded windows are excluded from attention.

`fine_tune='frozen'` updates only the recording aggregator and head. `fine_tune='last'` also
updates the final residual block and window embedding. No candidate is selected using an
outer-test participant.

In [ ]:
# ============================================================
# LP-ResMIL architecture and bounded random-search space
# ============================================================
def sepconv(filters,kernel_size,reg,padding='same',dilation_rate=1,name=None):
    """Keras 2/3-compatible separable convolution."""
    try:
        return layers.SeparableConv1D(filters,kernel_size,activation='linear',
            padding=padding,dilation_rate=dilation_rate,kernel_regularizer=reg,name=name)
    except (TypeError,ValueError):
        return layers.SeparableConv1D(filters,kernel_size,activation='linear',
            padding=padding,dilation_rate=dilation_rate,depthwise_regularizer=reg,
            pointwise_regularizer=reg,name=name)

def se_block(x,ratio,prefix):
    ch=int(x.shape[-1])
    s=layers.GlobalAveragePooling1D(name=prefix+'_squeeze')(x)
    s=layers.Dense(max(ch//ratio,4),activation='relu',name=prefix+'_reduce')(s)
    s=layers.Dense(ch,activation='sigmoid',name=prefix+'_expand')(s)
    return layers.Multiply(name=prefix+'_scale')([x,layers.Reshape((1,ch))(s)])

def build_window_encoder(cfg,n_ch=18):
    reg=keras.regularizers.l2(cfg['l2']); ch=cfg['channels']
    inp=layers.Input((WIN,n_ch),name='window_input')
    x=layers.SpatialDropout1D(cfg['spat_drop'],name='input_spatial_dropout')(inp)
    x=layers.GaussianNoise(0.01,name='input_gaussian_noise')(x)
    branches=[]
    for j,k in enumerate(cfg['kernels']):
        b=sepconv(max(ch//2,16),k,reg,name=f'multiscale_{j}_sep')(x)
        b=layers.BatchNormalization(name=f'multiscale_{j}_bn')(b)
        b=layers.Activation('elu',name=f'multiscale_{j}_elu')(b)
        branches.append(b)
    x=layers.Concatenate(name='multiscale_concat')(branches)
    x=layers.Conv1D(ch,1,padding='same',kernel_regularizer=reg,
                    name='multiscale_projection')(x)
    x=layers.BatchNormalization(name='multiscale_bn')(x)
    x=layers.Activation('elu',name='multiscale_elu')(x)
    x=se_block(x,cfg['se_ratio'],'multiscale_se')
    for block,dilation in enumerate((1,2,4)):
        prefix=f'block{block}'
        y=sepconv(ch,5,reg,dilation_rate=dilation,name=prefix+'_sep')(x)
        y=layers.BatchNormalization(name=prefix+'_bn')(y)
        y=layers.Activation('elu',name=prefix+'_pre_elu')(y)
        y=se_block(y,cfg['se_ratio'],prefix+'_se')
        x=layers.Add(name=prefix+'_add')([x,y])
        x=layers.Activation('elu',name=prefix+'_elu')(x)
    scores=layers.Dense(1,name='window_temporal_scores')(x)
    weights=layers.Softmax(axis=1,name='window_temporal_softmax')(scores)
    x=layers.Lambda(lambda t:tf.reduce_sum(t[0]*t[1],axis=1),
                    name='window_temporal_sum')([x,weights])
    x=layers.Dense(cfg['embedding_dim'],activation='elu',kernel_regularizer=reg,
                   name='window_embedding')(x)
    return keras.Model(inp,x,name='window_encoder')

def build_window_classifier(cfg):
    inp=layers.Input((WIN,18),name='window_classifier_input')
    encoder=build_window_encoder(cfg)
    x=encoder(inp)
    x=layers.Dropout(cfg['drop'],name='window_classifier_dropout')(x)
    out=layers.Dense(1,activation='sigmoid',
                     kernel_regularizer=keras.regularizers.l2(cfg['l2']),
                     name='window_pd_probability')(x)
    return keras.Model(inp,out,name='window_classifier')

def build_mil_model(window_model,cfg):
    """Masked gated attention over a variable number of windows per recording."""
    encoder=window_model.get_layer('window_encoder')
    mode=cfg['fine_tune']
    if mode=='frozen':
        encoder.trainable=False
    elif mode=='last':
        encoder.trainable=True
        for layer in encoder.layers:
            layer.trainable=(layer.name.startswith('block2_') or
                             layer.name in {'window_temporal_scores','window_embedding'})
    else:
        raise ValueError(f'Unknown fine_tune mode: {mode}')
    reg=keras.regularizers.l2(cfg['l2'])
    windows=layers.Input((None,WIN,18),name='recording_windows')
    mask=layers.Input((None,),dtype='bool',name='recording_window_mask')
    h=layers.TimeDistributed(encoder,name='shared_window_encoder')(windows)
    ah=max(cfg['embedding_dim']//2,16)
    a=layers.Dense(ah,activation='tanh',kernel_regularizer=reg,
                   name='recording_attention_tanh')(h)
    g=layers.Dense(ah,activation='sigmoid',kernel_regularizer=reg,
                   name='recording_attention_gate')(h)
    a=layers.Multiply(name='recording_attention_gated')([a,g])
    logits=layers.Dense(1,kernel_regularizer=reg,name='recording_attention_logits')(a)
    logits=layers.Lambda(lambda t:tf.squeeze(t,axis=-1),
                         name='recording_attention_squeeze')(logits)
    logits=layers.Lambda(lambda t:tf.where(t[1],t[0],tf.cast(-1e9,t[0].dtype)),
                         name='recording_attention_mask')([logits,mask])
    weights=layers.Softmax(axis=1,name='window_attention_weights')(logits)
    weights=layers.Lambda(lambda t:t[0]*tf.cast(t[1],t[0].dtype),
                          name='window_attention_remask')([weights,mask])
    z=layers.Lambda(lambda t:tf.reduce_sum(t[0]*tf.expand_dims(t[1],axis=-1),axis=1),
                    name='recording_embedding')([h,weights])
    z=layers.Dropout(cfg['drop'],name='recording_dropout')(z)
    out=layers.Dense(1,activation='sigmoid',kernel_regularizer=reg,
                     name='recording_pd_probability')(z)
    return keras.Model([windows,mask],out,name='LP_ResMIL')

SPACE_RESMIL=dict(channels=[32,48],kernels=[(3,7,15),(5,9,17)],
                  se_ratio=[4,8],embedding_dim=[32,64],spat_drop=[0.1,0.2],
                  drop=[0.25,0.4],l2=[1e-4,1e-3],fine_tune=['frozen','last'],
                  lr_mil=[1e-4,3e-4])

def parameter_report(cfg):
    window=build_window_classifier(cfg); full=build_mil_model(window,cfg)
    result=dict(window_parameters=int(window.count_params()),
                total_classifier_parameters=int(full.count_params()),
                trainable_parameters=int(sum(np.prod(w.shape) for w in full.trainable_weights)),
                lp_parameters=18*LP_ORDER,
                total_including_lp=int(full.count_params()+18*LP_ORDER))
    keras.backend.clear_session()
    return result

In [ ]:
# ============================================================
# Participant-balanced data batches and two-stage optimization
# ============================================================
def prepare_fold(train_idx,eval_idx):
    """Fit LP only on training controls; evaluation recordings are transformed only."""
    controls=[records[i]['proc'] for i in train_idx if records[i]['label']==0]
    if not controls: raise ValueError('LP fitting requires at least one training control.')
    C,stds=fit_lp(controls)
    return lp_features(train_idx+eval_idx,C,stds)

def recording_windows(i,feats):
    s=feats[i]
    starts=list(range(0,len(s)-WIN,STRIDE))  # exact-ending window excluded, as in LPGNet
    if not starts: raise ValueError(f'Recording {records[i]["file"]} is shorter than one window.')
    return np.stack([s[st:st+WIN] for st in starts]).astype(np.float32),starts

def recording_sample_weights(indices):
    """Weights sum to one per participant before mean normalization."""
    counts=Counter(records[i]['subject'] for i in indices)
    w=np.array([1.0/counts[records[i]['subject']] for i in indices],dtype=np.float32)
    return w/w.mean()

def window_dataset(indices,feats):
    """Windows inherit recording labels; weights give equal total mass per participant."""
    rec_w=recording_sample_weights(indices)
    X,y,w=[],[],[]
    for i,rw in zip(indices,rec_w):
        Xi,_=recording_windows(i,feats)
        X.append(Xi); y.extend([records[i]['label']]*len(Xi))
        w.extend([rw/len(Xi)]*len(Xi))
    X=np.concatenate(X).astype(np.float32)
    y=np.asarray(y,dtype=np.float32); w=np.asarray(w,dtype=np.float32)
    return X,y,w/w.mean()

def recording_batch(indices,feats):
    """Right-pad only the window dimension; boolean mask excludes every padded window."""
    arrays=[recording_windows(i,feats) for i in indices]
    max_n=max(len(a[0]) for a in arrays)
    X=np.zeros((len(indices),max_n,WIN,18),dtype=np.float32)
    mask=np.zeros((len(indices),max_n),dtype=bool)
    for j,(a,_) in enumerate(arrays):
        X[j,:len(a)]=a; mask[j,:len(a)]=True
    return [X,mask],labels(indices),recording_sample_weights(indices)

def callbacks():
    return [keras.callbacks.EarlyStopping('val_loss',patience=PATIENCE,
                                          restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau('val_loss',factor=0.25,
                                              patience=RLR_PATIENCE,min_lr=MIN_LR)]

def compile_window(model):
    model.compile(optimizer=keras.optimizers.Adam(LR_WIN,clipvalue=0.5),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=['accuracy'],weighted_metrics=[])

def compile_mil(model,cfg):
    model.compile(optimizer=keras.optimizers.Adam(cfg['lr_mil'],clipnorm=1.0),
                  loss=keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
                  metrics=['accuracy'],weighted_metrics=[])

def fit_resmil(cfg,core,early,feats,verbose=0):
    """Participant-disjoint early stopping; no outer-test data enter either stage."""
    window=build_window_classifier(cfg); compile_window(window)
    X,y,w=window_dataset(core,feats); Xv,yv,wv=window_dataset(early,feats)
    h1=window.fit(X,y,sample_weight=w,batch_size=BATCH_WIN,epochs=MAX_EPOCHS_WIN,
                  validation_data=(Xv,yv,wv),callbacks=callbacks(),verbose=verbose)
    full=build_mil_model(window,cfg); compile_mil(full,cfg)
    (R,yR,wR)=recording_batch(core,feats); (Rv,yRv,wRv)=recording_batch(early,feats)
    h2=full.fit(R,yR,sample_weight=wR,batch_size=BATCH_REC,epochs=MAX_EPOCHS_MIL,
                validation_data=(Rv,yRv,wRv),callbacks=callbacks(),verbose=verbose)
    best=dict(window=int(np.argmin(h1.history['val_loss'])+1),
              mil=int(np.argmin(h2.history['val_loss'])+1))
    return window,full,h1.history,h2.history,best

def refit_resmil(cfg,train_idx,feats,epochs,verbose=0):
    """Fixed-epoch refit on every outer-training participant; no validation or test data."""
    window=build_window_classifier(cfg); compile_window(window)
    X,y,w=window_dataset(train_idx,feats)
    window.fit(X,y,sample_weight=w,batch_size=BATCH_WIN,
               epochs=max(1,epochs['window']),verbose=verbose)
    full=build_mil_model(window,cfg); compile_mil(full,cfg)
    R,yR,wR=recording_batch(train_idx,feats)
    full.fit(R,yR,sample_weight=wR,batch_size=BATCH_REC,
             epochs=max(1,epochs['mil']),verbose=verbose)
    return window,full

def predict_recordings(full,indices,feats):
    out=[]
    for st in range(0,len(indices),PRED_BATCH):
        R,_,_=recording_batch(indices[st:st+PRED_BATCH],feats)
        out.extend(np.asarray(full.predict(R,verbose=0)).reshape(-1).tolist())
    return np.asarray(out,dtype=float)

def prediction_rows(indices,p,**base):
    return [dict(**base,file=records[i]['file'],subject=records[i]['subject'],
                 label=records[i]['label'],probability=float(prob))
            for i,prob in zip(indices,p)]

def attention_rows(full,indices,feats,**base):
    inspector=keras.Model(full.inputs,full.get_layer('window_attention_remask').output)
    rows=[]
    for st in range(0,len(indices),PRED_BATCH):
        batch=indices[st:st+PRED_BATCH]; R,_,_=recording_batch(batch,feats)
        weights=np.asarray(inspector.predict(R,verbose=0))
        for j,i in enumerate(batch):
            _,starts=recording_windows(i,feats)
            for k,start in enumerate(starts):
                rows.append(dict(**base,file=records[i]['file'],subject=records[i]['subject'],
                                 label=records[i]['label'],window=k,start_sample=start,
                                 end_sample=start+WIN,start_seconds=start/50.0,
                                 end_seconds=(start+WIN)/50.0,attention=float(weights[j,k])))
    return rows

In [ ]:
# ============================================================
# Fully nested automatic configuration and epoch selection
# ============================================================
def sample_unique_configs(space,n,rng):
    keys=list(space); max_n=int(np.prod([len(space[k]) for k in keys]))
    if n>max_n: raise ValueError('N_TRIALS exceeds the number of unique configurations.')
    out=[]; seen=set()
    while len(out)<n:
        cfg={k:space[k][int(rng.integers(len(space[k])))] for k in keys}
        token=json.dumps(cfg,sort_keys=True)
        if token not in seen: seen.add(token); out.append(cfg)
    return out

def inner_splits(outer_train,seed):
    ss=sorted({records[i]['subject'] for i in outer_train})
    yy=[subj_label[s] for s in ss]
    skf=StratifiedKFold(N_INNER,shuffle=True,random_state=seed)
    splits=[]
    for tr_s,va_s in skf.split(np.zeros(len(ss)),yy):
        tr_subj={ss[i] for i in tr_s}; va_subj={ss[i] for i in va_s}
        assert not tr_subj&va_subj and va_subj<=set(ss)
        tr=[i for i in outer_train if records[i]['subject'] in tr_subj]
        va=[i for i in outer_train if records[i]['subject'] in va_subj]
        splits.append((tr,va))
    return splits

def evaluate_inner_candidate(cfg,splits,seed,outer_fold,trial,feature_cache):
    rows=[]; epoch_rows=[]; start=time.time()
    for j,(tr,va) in enumerate(splits):
        core,early=split_val(tr,seed=SPLIT_SEED+1000+j)
        # LP sees core only: neither early-stopping nor candidate-validation participants fit it.
        if j not in feature_cache: feature_cache[j]=prepare_fold(core,early+va)
        feats=feature_cache[j]
        keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed+10000+j)
        window,full,h1,h2,best=fit_resmil(cfg,core,early,feats)
        p=predict_recordings(full,va,feats)
        rows+=prediction_rows(va,p,family='LP_ResMIL',seed=seed,outer_fold=outer_fold,
                              trial=trial,inner_fold=j,fold=j)
        epoch_rows.append(best)
    inner=oof_frame(rows)
    part=participant_oof(inner,['family','seed','outer_fold','trial'])
    metrics=metric_dict(part['label'],part['probability'])
    return metrics,inner,epoch_rows,time.time()-start

def nested_search(space,outer_fold,outer_train,seed):
    # Same candidates and inner splits across training seeds; fitted initializations differ.
    rng=np.random.default_rng(SEARCH_SEED+outer_fold)
    candidates=sample_unique_configs(space,N_TRIALS,rng)
    splits=inner_splits(outer_train,SPLIT_SEED+777*outer_fold)
    trials=[]; selected_inner=[]; best_idx=None; best_key=(np.inf,-np.inf)
    feature_cache={}
    for t,cfg in enumerate(candidates):
        metrics,inner,epochs,seconds=evaluate_inner_candidate(
            cfg,splits,seed,outer_fold,t,feature_cache)
        params=parameter_report(cfg)
        row=dict(family='LP_ResMIL',seed=seed,outer_fold=outer_fold,trial=t,candidate=cfg,
                 inner_participant_log_loss=metrics['log_loss'],
                 inner_participant_auc=metrics['roc_auc'],
                 inner_participant_brier=metrics['brier'],
                 inner_participant_accuracy=metrics['accuracy'],
                 mean_best_window_epoch=float(np.mean([e['window'] for e in epochs])),
                 mean_best_mil_epoch=float(np.mean([e['mil'] for e in epochs])),
                 sd_best_window_epoch=float(np.std([e['window'] for e in epochs])),
                 sd_best_mil_epoch=float(np.std([e['mil'] for e in epochs])),
                 training_seconds=seconds,**params)
        trials.append(row)
        key=(metrics['log_loss'],-metrics['roc_auc'])
        if key<best_key:
            best_key=key; best_idx=t; selected_inner=inner
        print(f"[LP-ResMIL] outer {outer_fold} seed {seed} trial {t:02d} | "
              f"inner NLL={metrics['log_loss']:.4f} AUC={metrics['roc_auc']:.3f} | {cfg}")
    return best_idx,trials,selected_inner

def fit_selected_outer(cfg,outer_fold,seed):
    f=FOLDS[outer_fold]; tr_all,te=f['train'],f['test']
    assert not ({records[i]['subject'] for i in tr_all}&{records[i]['subject'] for i in te})
    core,early=split_val(tr_all,seed=SPLIT_SEED+300000+outer_fold)
    early_feats=prepare_fold(core,early)  # early-stopping participants never fit LP
    keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed+400000)
    _,_,h1,h2,best=fit_resmil(cfg,core,early,early_feats)
    # After epoch selection, refit LP and both stages on all outer-training participants.
    feats=prepare_fold(tr_all,te)
    keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed+400000)
    window,full=refit_resmil(cfg,tr_all,feats,best)
    p=predict_recordings(full,te,feats)
    return p,best,full,feats

In [ ]:
# ============================================================
# Run only LP-ResMIL; checkpoint tables after every outer task
# ============================================================
all_oof_rows=[]; all_trials=[]; selections=[]; selected_inner_tables=[]; all_attention_rows=[]
OUTDIR.mkdir(exist_ok=True)
t0=time.time()
for seed in TRAIN_SEEDS:
    for k in range(N_FOLDS):
        best_idx,trials,inner=nested_search(SPACE_RESMIL,k,FOLDS[k]['train'],seed)
        selected=trials[best_idx]; cfg=selected['candidate']
        all_trials.extend(trials); selected_inner_tables.append(inner.assign(selected=True))
        p,best,full,feats=fit_selected_outer(cfg,k,seed)
        all_oof_rows+=prediction_rows(FOLDS[k]['test'],p,family='LP_ResMIL',seed=seed,fold=k)
        selections.append(dict(family='LP_ResMIL',seed=seed,outer_fold=k,candidate=cfg,
            best_window_epoch=best['window'],best_mil_epoch=best['mil'],
            inner_participant_log_loss=selected['inner_participant_log_loss'],
            inner_participant_auc=selected['inner_participant_auc'],
            window_parameters=selected['window_parameters'],
            total_classifier_parameters=selected['total_classifier_parameters'],
            trainable_parameters=selected['trainable_parameters'],
            lp_parameters=selected['lp_parameters'],
            total_including_lp=selected['total_including_lp']))
        if EXPORT_ATTENTION:
            all_attention_rows+=attention_rows(full,FOLDS[k]['test'],feats,
                                               family='LP_ResMIL',seed=seed,fold=k)
        print(f'[LP-ResMIL] outer {k} seed {seed} SELECTED {cfg}')
        # Partial outputs make an interrupted long run inspectable; they are not final results.
        pd.DataFrame(all_oof_rows).to_csv(OUTDIR/'checkpoint_recording_oof.csv',index=False)
        pd.DataFrame(all_trials).to_csv(OUTDIR/'checkpoint_search_trials.csv',index=False)
        pd.DataFrame(selections).to_csv(OUTDIR/'checkpoint_selected_configurations.csv',index=False)
        keras.backend.clear_session()

nested_oof=oof_frame(all_oof_rows)
trial_table=pd.DataFrame(all_trials)
selection_table=pd.DataFrame(selections)
inner_selected_oof=pd.concat(selected_inner_tables,ignore_index=True)
attention_oof=pd.DataFrame(all_attention_rows)
print(f'All LP-ResMIL experiments completed in {(time.time()-t0)/3600:.2f} hours')
show_table(selection_table,'Selected configuration independently inside every outer fold',limit=100)
show_table(trial_table,'Every inner-search trial; no outer-test selection',limit=100)

In [ ]:
# ============================================================
# Complete OOF performance, calibration, and uncertainty
# ============================================================
nested_summary=summarize_oof(nested_oof,['family','seed'])
show_table(nested_summary.round(4),'Complete OOF metrics by training seed')
nested_ci=pooled_bootstrap(nested_oof,['family','seed'])
show_table(nested_ci.round(4),
           'Participant OOF metrics and 95% stratified bootstrap intervals',limit=100)
participant_cols=[c for c in nested_summary.columns
                  if c.startswith('participant_') and c!='participant_n']
across_seed=nested_summary.groupby('family')[participant_cols].agg(['mean','std']).round(4)
show_table(across_seed,'Training-seed variability on identical outer folds; SD is not a CI')

nested_fold_rows=[]
for (family,seed,fold),g in nested_oof.groupby(['family','seed','fold']):
    nested_fold_rows.append(dict(family=family,seed=seed,fold=fold,
                                 **metric_dict(g['label'],g['probability'])))
nested_fold_metrics=pd.DataFrame(nested_fold_rows)
show_table(nested_fold_metrics.round(4),
           'Descriptive outer-fold recording metrics; overlapping training folds are not independent',limit=100)
plot_oof_diagnostics(nested_oof,['family','seed'])

In [ ]:
# ============================================================
# Optional paired comparison with completed prior OOF predictions
# ============================================================
def paired_reference_bootstrap(current,reference,ref_family,
        metrics=('accuracy','balanced_accuracy','macro_f1','sensitivity','specificity',
                 'roc_auc','log_loss','brier')):
    """LP-ResMIL minus reference; averages seed-specific paired effects, not an ensemble."""
    ref=reference[reference['family']==ref_family].copy()
    seeds=sorted(set(current['seed'])&set(ref['seed']))
    if not seeds: raise ValueError(f'No common training seeds for {ref_family}.')
    aligned=[]
    for seed in seeds:
        cur=participant_oof(current[current['seed']==seed],['family','seed'])
        old=participant_oof(ref[ref['seed']==seed],['family','seed'])
        if set(cur['subject'])!=set(old['subject']):
            raise ValueError(f'{ref_family} seed {seed}: reference is incomplete or uses another cohort.')
        joined=cur[['subject','label','fold','probability']].merge(
            old[['subject','label','fold','probability']],on='subject',suffixes=('_cur','_ref'),
            validate='one_to_one')
        if not np.array_equal(joined['label_cur'],joined['label_ref']):
            raise ValueError('Diagnosis mismatch in reference OOF predictions.')
        if not np.array_equal(joined['fold_cur'],joined['fold_ref']):
            raise ValueError('Outer-fold mismatch; paired comparison is not valid.')
        aligned.append((joined['label_cur'].to_numpy(),joined['probability_cur'].to_numpy(),
                        joined['probability_ref'].to_numpy()))
    y=aligned[0][0]; classes=[np.flatnonzero(y==c) for c in (0,1)]
    rows=[]
    for metric in metrics:
        observed=float(np.mean([metric_dict(y,a)[metric]-metric_dict(y,b)[metric]
                                for y,a,b in aligned]))
        rng=np.random.default_rng(2028+sum(ord(c) for c in metric+ref_family))
        vals=[]
        for _ in range(N_BOOT):
            ii=np.concatenate([rng.choice(ix,len(ix),replace=True) for ix in classes])
            vals.append(np.mean([metric_dict(y[ii],a[ii])[metric]-
                                 metric_dict(y[ii],b[ii])[metric] for y,a,b in aligned]))
        rows.append(dict(comparison=f'LP_ResMIL minus {ref_family}',seeds_used=str(seeds),
            metric=metric,seed_averaged_difference=observed,
            bootstrap_95_low=np.nanpercentile(vals,2.5),
            bootstrap_95_high=np.nanpercentile(vals,97.5)))
    return pd.DataFrame(rows)

paired_comparisons=pd.DataFrame()
if REFERENCE_OOF_PATH.exists():
    reference_oof=pd.read_csv(REFERENCE_OOF_PATH)
    comparisons=[]
    for family in [f for f in ('baseline','A','B') if f in set(reference_oof['family'])]:
        try:
            comparisons.append(paired_reference_bootstrap(nested_oof,reference_oof,family))
        except ValueError as exc:
            print(f'Reference comparison skipped for {family}: {exc}')
    if comparisons:
        paired_comparisons=pd.concat(comparisons,ignore_index=True)
        show_table(paired_comparisons.round(4),
            'Paired participant differences; negative log loss/Brier favors LP-ResMIL',limit=100)
else:
    print('No completed reference OOF file found; no prior model was retrained and no comparison was fabricated.')

In [ ]:
# ============================================================
# Participant errors, HY severity, study patterns, and stability
# ============================================================
nested_participants,nested_errors,nested_pd_severity,nested_study=error_tables(
    nested_oof,['family','seed'])
show_table(nested_errors[['family','seed','fold','subject','study','diagnosis','hy_label','age',
    'recordings','probability','prediction','confidence','nll']].round(4),
    'All misclassified outer-test participants',limit=250)
show_table(nested_pd_severity.round(4),
    'PD sensitivity by verified HY stage; Unknown retained and controls never assigned HY 0')
show_table(nested_study.round(4),'Exploratory participant OOF metrics by study and diagnosis',limit=100)

seed_stability=(nested_participants.groupby(['family','subject']).agg(
    error_seeds=('correct',lambda x:int((x==0).sum())),n_seeds=('seed','nunique'),
    mean_probability=('probability','mean'),mean_nll=('nll','mean')).reset_index())
seed_stability=seed_stability.merge(nested_participants[
    ['subject','study','diagnosis','hy_label','age']].drop_duplicates('subject'),
    on='subject',how='left')
show_table(seed_stability.sort_values(['error_seeds','mean_nll'],ascending=[False,False]).round(4),
           'Persistent versus seed-variable participant errors',limit=200)

inner_loss=participant_oof(inner_selected_oof,['family','seed','outer_fold','trial'])
show_table(inner_loss.sort_values('nll',ascending=False)[['family','seed','outer_fold','subject',
    'diagnosis','hy_label','probability','prediction','nll']].head(50).round(4),
    'Highest-NLL selected inner participants; repetitions across outer folds are not independent')

attention_audit=pd.DataFrame()
if not attention_oof.empty:
    attention_audit=(attention_oof.groupby(['family','seed','fold','file','subject'],as_index=False)
        .agg(valid_windows=('window','count'),attention_sum=('attention','sum'),
             maximum_window_attention=('attention','max')))
    attention_audit['absolute_sum_error']=(attention_audit['attention_sum']-1).abs()
    show_table(attention_audit.round(6),
        'Masked-attention audit: every recording should have attention sum 1',limit=100)
    top_attention=(attention_oof.sort_values(['seed','fold','file','attention'],
        ascending=[True,True,True,False]).groupby(['seed','fold','file'],group_keys=False)
        .head(3))
    show_table(top_attention.round(5),'Three highest-attention valid windows per test recording',limit=100)

In [ ]:
# ============================================================
# Final reproducibility, predictions, selection, and error exports
# ============================================================
OUTDIR.mkdir(exist_ok=True)
exports=[('recording_oof',nested_oof),('participant_oof',nested_participants),
    ('pooled_metrics',nested_summary),('participant_bootstrap',nested_ci),
    ('fold_metrics',nested_fold_metrics),('paired_comparisons',paired_comparisons),
    ('search_trials',trial_table),('selected_configurations',selection_table),
    ('participant_errors',nested_errors),('pd_severity',nested_pd_severity),
    ('study_metrics',nested_study),('seed_stability',seed_stability),
    ('selected_inner_oof',inner_selected_oof),('attention_weights',attention_oof),
    ('attention_audit',attention_audit)]
for name,df in exports: df.to_csv(OUTDIR/f'{name}.csv',index=False)

fold_assignments=pd.DataFrame([
    dict(outer_fold=k,partition='train' if i in set(f['train']) else 'test',
         file=records[i]['file'],subject=records[i]['subject'],label=records[i]['label'])
    for k,f in enumerate(FOLDS) for i in f['train']+f['test']])
fold_assignments.to_csv(OUTDIR/'fold_assignments.csv',index=False)
data_inventory=pd.DataFrame([dict(file=r['file'],study=r['study'],subject=r['subject'],
    label=r['label'],raw_samples=len(r['signal']),processed_samples=len(r['proc'])) for r in records])
data_inventory.to_csv(OUTDIR/'data_inventory.csv',index=False)

protocol=dict(model='LP-ResMIL',outer_folds=N_FOLDS,inner_folds=N_INNER,
    fold_seed=FOLD_SEED,split_seed=SPLIT_SEED,search_seed=SEARCH_SEED,
    training_seeds=TRAIN_SEEDS,unique_candidates_per_outer_fold_and_seed=N_TRIALS,
    search_space=SPACE_RESMIL,selection_metric='pooled inner participant log loss',
    tie_breaker='inner participant AUC',threshold=THRESHOLD,
    window_samples=WIN,window_stride=STRIDE,sampling_rate_hz=50,lp_order=LP_ORDER,
    participant_balanced_window_loss=True,participant_balanced_recording_loss=True,
    recording_aggregation='masked gated attention MIL',
    participant_aggregation='arithmetic mean of recording probabilities',
    final_refit='all outer-training participants with outer-training-selected epochs',
    reference_models_retrained=False,reference_oof_path=str(REFERENCE_OOF_PATH),
    bootstrap_iterations=N_BOOT,bootstrap_unit='participant, stratified by diagnosis',
    tensorflow_version=tf.__version__,numpy_version=np.__version__,pandas_version=pd.__version__,
    scipy_version=scipy.__version__,sklearn_version=sklearn.__version__,
    python_version=platform.python_version(),platform=platform.platform())
(OUTDIR/'protocol.json').write_text(json.dumps(protocol,indent=2))
print('Final CSV and protocol outputs written to:',OUTDIR.resolve())

## Reporting checklist and limitations

- Report all 10 outer folds, all declared seeds, every selected configuration, and the complete
  candidate-search table. Do not select the best seed or best outer fold.
- Participant OOF metrics are primary; recording and window results are secondary. Repeated
  seeds do not create additional independent participants.
- Bootstrap intervals are conditional on fitted OOF predictions. They exclude uncertainty from
  model refitting, new fold assignments, external-site shift, and future populations.
- Paired reference comparisons require completed OOF files from the same participant cohort,
  outer folds, labels, and common training seeds. The notebook validates these conditions.
- If the original LPGNet and Variant A both outperform Variant B, this supports discarding the
  **evaluated control-trained GRU residual for this dataset and protocol**. It does not prove
  that every nonlinear residual or every GRU design is ineffective.
- Attention weights describe allocation within the fitted recording model. They are not causal
  explanations, validated gait-phase localizations, or independent clinical evidence.
- HY and study analyses are exploratory. PD sensitivity within an HY stratum is not severity-
  classification accuracy, and controls are never assigned HY 0.
- This architecture was motivated by earlier experiments on the same dataset. Nested selection
  prevents outer-test use within this run, but external validation remains the strongest check
  against adaptive researcher-level overfitting.